In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 89.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: 

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import torch
from torch.utils.data import DataLoader, Dataset


# Load datasets
train_df = pd.read_csv('/kaggle/input/dataset-rrck/Train_RRCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-rrck/Test_RRCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("seyonec/PubChem10M_SMILES_BPE_450k")
model = AutoModelForSequenceClassification.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/336M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/PubChem10M_SMILES_BPE_450k and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }

In [5]:
# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [8]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 9/9 [00:04<00:00,  2.13batch/s]


Epoch 1/20 - Train Loss: 9.2065
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 9/9 [00:03<00:00,  2.42batch/s]


Epoch 2/20 - Train Loss: 0.6164
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.1145
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 9/9 [00:04<00:00,  2.15batch/s]


Epoch 19/20 - Train Loss: 0.1067
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 9/9 [00:04<00:00,  2.12batch/s]

Epoch 20/20 - Train Loss: 0.0931


In [9]:
# Saving the model after training
model_name = 'PubChem10M_SMILES_BPE_450k_model_1_rrck'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_rrck


In [10]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)

# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 3/3 [00:00<00:00,  8.03batch/s]

Test Loss: 0.2814
(35,)
(35,)
Mean Squared Error: 0.3661
Root Mean Squared Error: 0.6050
Mean Absolute Error: 0.4266
R^2 Score: 0.2074
Pearson Correlation Coefficient: 0.6236
Spearman Correlation Coefficient: 0.6067
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [11]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'PubChem10M_SMILES_BPE_450k_model_1_rrck'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_save_path, trust_remote_code=True)
model = AutoModel.from_pretrained(model_save_path, trust_remote_code=True).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_rrck and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/dataset-rrck/Train_RRCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-rrck/Test_RRCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [13]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [14]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)


In [15]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 9/9 [00:00<00:00, 12.00it/s]


torch.Size([140, 198, 768])
torch.Size([140, 768])


In [16]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [17]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 3/3 [00:00<00:00, 15.94it/s]

torch.Size([35, 201, 768])
torch.Size([35, 768])


In [18]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [19]:
train_data.to_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv",index=False)
test_data.to_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv",index=False)

In [20]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [21]:
train_data = pd.read_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv")
test_data = pd.read_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv")

In [22]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [23]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=42),
    DecisionTreeRegressor(random_state=42),
    RandomForestRegressor(n_jobs=-1, random_state=42),
    GradientBoostingRegressor(random_state=42),
    AdaBoostRegressor(random_state=42),
    xgb.XGBRegressor(random_state=42),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=42),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=42)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 768)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 768)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002600 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 29952
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 768
[LightGBM] [Info] Start training from score -5.679129
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0894,0.2248,0.2991,0.7715,0.8788,0.8865,0.2617,0.3567,0.5116,0.4334,0.6810,0.6514
DecisionTreeRegressor,0.1771,0.3046,0.4209,0.5474,0.7626,0.7854,0.2616,0.3673,0.5115,0.4336,0.6938,0.6580
RandomForestRegressor,0.0839,0.2114,0.2896,0.7857,0.8875,0.8971,0.2637,0.3539,0.5135,0.4291,0.6750,0.6429
GradientBoostingRegressor,0.0883,0.2142,0.2972,0.7744,0.8800,0.8855,0.2457,0.3423,0.4957,0.4681,0.7050,0.6730
AdaBoostRegressor,0.0882,0.2182,0.2970,0.7747,0.8811,0.8905,0.2518,0.3561,0.5018,0.4548,0.6944,0.6441
XGBRegressor,0.0937,0.2295,0.3061,0.7606,0.8723,0.8801,0.2404,0.3502,0.4904,0.4794,0.7130,0.6826
ExtraTreesRegressor,0.0933,0.2238,0.3054,0.7617,0.8736,0.8845,0.2441,0.3412,0.4941,0.4715,0.7033,0.6709
LinearRegression,0.4102,0.4845,0.6405,-0.0480,0.6986,0.6998,0.3838,0.4484,0.6195,0.1691,0.6263,0.5738
KNeighborsRegressor,0.1244,0.2514,0.3527,0.6821,0.8296,0.8439,0.2659,0.3452,0.5157,0.4242,0.6838,0.6417
SVR,0.0952,0.2247,0.3086,0.7567,0.8702,0.8819,0.2601,0.3421,0.5100,0.4369,0.6883,0.6298


In [24]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0894,0.2248,0.2991,0.7715,0.8788,0.8865,0.2617,0.3567,0.5116,0.4334,0.6810,0.6514
DecisionTreeRegressor,0.1771,0.3046,0.4209,0.5474,0.7626,0.7854,0.2616,0.3673,0.5115,0.4336,0.6938,0.6580
RandomForestRegressor,0.0839,0.2114,0.2896,0.7857,0.8875,0.8971,0.2637,0.3539,0.5135,0.4291,0.6750,0.6429
GradientBoostingRegressor,0.0883,0.2142,0.2972,0.7744,0.8800,0.8855,0.2457,0.3423,0.4957,0.4681,0.7050,0.6730
AdaBoostRegressor,0.0882,0.2182,0.2970,0.7747,0.8811,0.8905,0.2518,0.3561,0.5018,0.4548,0.6944,0.6441
XGBRegressor,0.0937,0.2295,0.3061,0.7606,0.8723,0.8801,0.2404,0.3502,0.4904,0.4794,0.7130,0.6826
ExtraTreesRegressor,0.0933,0.2238,0.3054,0.7617,0.8736,0.8845,0.2441,0.3412,0.4941,0.4715,0.7033,0.6709
LinearRegression,0.4102,0.4845,0.6405,-0.0480,0.6986,0.6998,0.3838,0.4484,0.6195,0.1691,0.6263,0.5738
KNeighborsRegressor,0.1244,0.2514,0.3527,0.6821,0.8296,0.8439,0.2659,0.3452,0.5157,0.4242,0.6838,0.6417
SVR,0.0952,0.2247,0.3086,0.7567,0.8702,0.8819,0.2601,0.3421,0.5100,0.4369,0.6883,0.6298


In [25]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.9608277681986035, -6.584186568314052, -6.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.587890133761851, -6.005813022906539, -6.6...","[-6.4177566131953725, -5.910171627213067, -6.5...","[0.12220784435153856, 0.07678498517556663, 0.1..."
1,DecisionTreeRegressor,"[-5.76, -6.46, -6.93, -5.01, -5.39, -6.02, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.66, -5.76, -7.0, -6.89, -6.66, -5.02, -5....","[-6.470000000000001, -5.834, -7.0, -6.866, -6....","[0.4515750214526929, 0.14800000000000005, 0.0,..."
2,RandomForestRegressor,"[-5.8971999999999944, -6.543699999999998, -6.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.475899999999995, -5.959899999999996, -6.6...","[-6.3199099999999975, -5.903109999999998, -6.5...","[0.11288581133162698, 0.043014292973382426, 0...."
3,GradientBoostingRegressor,"[-5.783385593356304, -6.532362793944429, -6.52...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.816220371918318, -5.775148000066602, -6.8...","[-6.535545346455356, -5.821750570748927, -6.66...","[0.15218247260253723, 0.04822404596387102, 0.1..."
4,AdaBoostRegressor,"[-5.846666666666666, -6.5218181818181815, -6.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.5922222222222215, -5.8268, -6.88500000000...","[-6.330440656565656, -5.825027473118279, -6.74...","[0.1498189921756883, 0.05961749587820472, 0.08..."
5,XGBRegressor,"[-5.7385263, -6.542835, -6.2034416, -5.2503843...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.669117, -5.7650123, -6.619759, -6.788895,...","[-6.499949, -5.86629, -6.7166734, -6.7422166, ...","[0.115413785, 0.08316421, 0.1733997, 0.1034006..."
6,ExtraTreesRegressor,"[-5.844899999999995, -6.5360000000000005, -6.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.478149999999996, -5.799399999999994, -6.6...","[-6.407589999999997, -5.824929999999997, -6.55...","[0.068131610871899, 0.028285925828937283, 0.09..."
7,LinearRegression,"[-6.0703554170353105, -7.006646267754708, -7.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.555711702501415, -5.676381941238338, -4.9...","[-6.76073199272241, -5.78243972281564, -5.3437...","[0.2997421563690203, 0.11306800324675442, 0.58..."
8,KNeighborsRegressor,"[-6.183333333333334, -6.46, -6.963333333333334...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.183333333333334, -6.183333333333334, -6.4...","[-6.076, -6.076, -6.237333333333333, -6.441333...","[0.13745948898170432, 0.13745948898170432, 0.2..."
9,SVR,"[-6.041575898455863, -6.411143742330919, -6.73...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.2495840513732634, -6.101195250023864, -6....","[-6.164622927761784, -6.038178603322683, -6.55...","[0.05251342725068136, 0.03283091867757459, 0.1..."


In [26]:
result_df.to_csv('/kaggle/working/Results_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_rrck.csv')